# Canada's Economic Pulse
### Are Canadians keeping up? Wages, Housing, and Employment (2014–2024)

**Data sources:** Statistics Canada open data (tables 14-10-0023-01, 18-10-0005-01, 14-10-0064-01)  
**Author:** Muhammad  

---
**Before running:** make sure you've run `data/fetch_data.py` and `data/clean_data.py` first.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

PROC = 'processed/'

wages_vs_cpi   = pd.read_csv(PROC + 'wages_vs_cpi.csv')
wages_annual   = pd.read_csv(PROC + 'wages_annual.csv')
cpi_annual     = pd.read_csv(PROC + 'cpi_annual.csv')
emp_annual     = pd.read_csv(PROC + 'employment_annual.csv')
emp_monthly    = pd.read_csv(PROC + 'employment_monthly.csv')

print('Datasets loaded ✓')
print(f'  wages_vs_cpi:  {wages_vs_cpi.shape}')
print(f'  wages_annual:  {wages_annual.shape}')
print(f'  cpi_annual:    {cpi_annual.shape}')
print(f'  emp_annual:    {emp_annual.shape}')

FileNotFoundError: [Errno 2] No such file or directory: 'data/processed/wages_vs_cpi.csv'

## 1. Quick data inspection

In [ ]:
wages_vs_cpi.head(10)

In [ ]:
# Summary stats for wages
wages_annual.groupby('year')['avg_hourly_wage'].describe().round(2)

## 2. Chart 1 — Wages vs. Shelter CPI (indexed to 2014 = 100)

> **The core story:** If shelter costs rise faster than wages, Canadians are falling behind on housing affordability.

In [ ]:
# National view first
national = wages_vs_cpi[wages_vs_cpi['geo'] == 'Canada'].copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=national['year'], y=national['wage_index'],
    name='Avg Hourly Wage Index',
    line=dict(color='#2196F3', width=2.5)
))
fig.add_trace(go.Scatter(
    x=national['year'], y=national['shelter_index'],
    name='Shelter CPI Index',
    line=dict(color='#F44336', width=2.5, dash='dash')
))
fig.add_trace(go.Scatter(
    x=national['year'], y=national['cpi_index'],
    name='All-Items CPI Index',
    line=dict(color='#FF9800', width=1.5, dash='dot')
))

fig.add_hline(y=100, line_dash='solid', line_color='gray', opacity=0.4)

fig.update_layout(
    title='Wages vs. Housing Costs in Canada (2014 = 100)',
    xaxis_title='Year',
    yaxis_title='Index (2014 = 100)',
    legend=dict(x=0.01, y=0.99),
    height=450,
    template='plotly_white'
)
fig.show()

In [ ]:
# ── Finding: Gap between shelter and wages in 2024
row_2024 = national[national['year'] == 2024].iloc[0]
gap = row_2024['shelter_index'] - row_2024['wage_index']
print(f"In 2024 (indexed to 2014=100):")
print(f"  Wage index:    {row_2024['wage_index']:.1f}")
print(f"  Shelter index: {row_2024['shelter_index']:.1f}")
print(f"  Gap:           +{gap:.1f} points — shelter outpaced wages by {gap:.1f}%")

## 3. Chart 2 — Provincial comparison: Wage growth vs. Shelter CPI (2024)

> Which provinces are hit hardest?

In [ ]:
PROVINCES = [
    'British Columbia', 'Alberta', 'Saskatchewan', 'Manitoba',
    'Ontario', 'Quebec', 'New Brunswick', 'Nova Scotia',
    'Prince Edward Island', 'Newfoundland and Labrador'
]

prov_2024 = wages_vs_cpi[
    (wages_vs_cpi['year'] == 2024) &
    (wages_vs_cpi['geo'].isin(PROVINCES))
].copy()

prov_2024['affordability_gap'] = prov_2024['shelter_index'] - prov_2024['wage_index']
prov_2024 = prov_2024.sort_values('affordability_gap', ascending=True)

fig2 = go.Figure()

fig2.add_trace(go.Bar(
    name='Wage Index (2024)',
    y=prov_2024['geo'],
    x=prov_2024['wage_index'],
    orientation='h',
    marker_color='#2196F3'
))
fig2.add_trace(go.Bar(
    name='Shelter CPI Index (2024)',
    y=prov_2024['geo'],
    x=prov_2024['shelter_index'],
    orientation='h',
    marker_color='#F44336',
    opacity=0.7
))

fig2.update_layout(
    title='Wage Growth vs. Shelter Cost Growth by Province (2024, indexed to 2014=100)',
    xaxis_title='Index (2014 = 100)',
    barmode='overlay',
    height=500,
    template='plotly_white'
)
fig2.show()

print("\nAffordability gap (Shelter index - Wage index):")
print(prov_2024[['geo', 'wage_index', 'shelter_index', 'affordability_gap']]
      .sort_values('affordability_gap', ascending=False)
      .to_string(index=False))

## 4. Chart 3 — Employment by industry over time

> Which sectors have grown or shrunk since 2014?

In [ ]:
# Focus on top industries by 2019 employment (pre-COVID baseline)
top_industries = (
    emp_annual[emp_annual['year'] == 2019]
    .nlargest(8, 'avg_employed_thousands')['industry']
    .tolist()
)

emp_top = emp_annual[emp_annual['industry'].isin(top_industries)].copy()

fig3 = px.line(
    emp_top,
    x='year',
    y='avg_employed_thousands',
    color='industry',
    title='Employment by Industry in Canada (2014–2024, thousands)',
    labels={
        'avg_employed_thousands': 'Avg Employed (thousands)',
        'year': 'Year',
        'industry': 'Industry'
    },
    template='plotly_white',
    height=500
)

# Mark COVID shock
fig3.add_vline(x=2020, line_dash='dash', line_color='red', opacity=0.5,
               annotation_text='COVID-19', annotation_position='top right')

fig3.show()

In [ ]:
# Recovery check: compare 2024 employment to 2019 pre-COVID levels
pre_covid  = emp_annual[emp_annual['year'] == 2019].set_index('industry')['avg_employed_thousands']
post_covid = emp_annual[emp_annual['year'] == 2024].set_index('industry')['avg_employed_thousands']

recovery = ((post_covid - pre_covid) / pre_covid * 100).dropna().sort_values(ascending=False)
print("Employment change 2019 → 2024 (%):")
print(recovery.round(1).to_string())

## 5. Summary of findings

*(Fill this in after running the charts — write 3–5 bullet points of what you actually found. This becomes your README summary and your interview talking point.)*

- **Finding 1:** Shelter CPI outpaced wage growth nationally by approximately ___ index points between 2014 and 2024.
- **Finding 2:** The provinces with the largest affordability gaps were ___ and ___.
- **Finding 3:** The industries that recovered most strongly post-COVID were ___.
- **Finding 4:** The industries that are still below 2019 employment levels include ___.
- **Overall:** Housing affordability has deteriorated across all provinces, with wage growth consistently lagging shelter cost increases.